In [1]:
# 1. Install OpenBabel chemistry toolkit
!apt-get install openbabel -y

# 2. Download the official AutoDock Vina 1.2.5 executable
!wget https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina
!chmod +x vina

print("\n==== SETUP COMPLETE: OpenBabel and Vina are ready! ====")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 5 newly installed, 0 to remove and 3 not upgraded.
Need to get 4,148 kB of archives.
After this operation, 19.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-iostreams1.74.0 amd64 1.74.0-14ubuntu3 [245 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libinchi1 amd64 1.03+dfsg-4 [455 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmaeparser1 amd64 1.2.4-1build1 [88.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenbabel7 amd64 3.1.1+dfsg-6ubuntu5 [3,231 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openbabel amd

In [2]:
# 1. Download the Aromatase crystal structure
!wget https://files.rcsb.org/download/3EQM.pdb

# 2. Prepare the receptor using OpenBabel (-as adds hydrogens and charges)
!obabel 3EQM.pdb -O receptor.pdbqt -as

print("\n==== RECEPTOR READY: receptor.pdbqt created successfully! ====")

--2026-06-30 15:42:07--  https://files.rcsb.org/download/3EQM.pdb
Resolving files.rcsb.org (files.rcsb.org)... 18.154.206.26, 18.154.206.22, 18.154.206.44, ...
Connecting to files.rcsb.org (files.rcsb.org)|18.154.206.26|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘3EQM.pdb’

3EQM.pdb                [ <=>                ] 347.02K  --.-KB/s    in 0.03s   

2026-06-30 15:42:07 (12.5 MB/s) - ‘3EQM.pdb’ saved [355347]

1 molecule converted

==== RECEPTOR READY: receptor.pdbqt created successfully! ====


In [6]:
import subprocess
import glob
import os

# Define the chemical structures (SMILES)
ligand_library = {
    "Baicalein": "O=C1C=C(C2=CC=CC=C2)OC3=CC(O)=C(O)C(O)=C13",
    "Chrysin": "O=C1C=C(C2=CC=CC=C2)OC3=CC(O)=CC(O)=C13",
    "Oroxylin_A": "COC1=C(O)C=C2C(=C1O)C(=O)C=C(O2)C3=CC=CC=C3",
    "Exemestane_Control": "CC12CCC3C(C1CCC2=O)CC(=C)C4=CC(=O)C=CC34C"
}

print("=== 🛠️ STEP 1: GENERATING 3D LIGAND STRUCTURES ===")
for name, smiles in ligand_library.items():
    pdb_file = f"{name}.pdb"
    pdbqt_file = f"{name}_expanded.pdbqt"

    # Standard echo piping ensures cross-platform compatibility for SMILES strings
    subprocess.run(f'echo "{smiles}" | obabel -ismi -O {pdb_file} --gen3d', shell=True)
    subprocess.run(f'obabel {pdb_file} -O {pdbqt_file} -as', shell=True)

    if os.path.exists(pdbqt_file) and os.path.getsize(pdbqt_file) > 0:
        print(f"   ✓ Successfully generated and verified: {pdbqt_file}")
    else:
        print(f"   ❌ Error: Failed to build structure for {name}")

print("\n=== 🚀 STEP 2: RUNNING PHYSICS-BASED DOCKING LOOP ===")
center_x, center_y, center_z = 85.027, 54.737, 46.428
size_x, size_y, size_z = 22.0, 22.0, 22.0

expanded_hits = glob.glob("*_expanded.pdbqt")
print(f"Found {len(expanded_hits)} target ligands ready for simulation.\n")

for ligand in expanded_hits:
    output_name = ligand.replace(".pdbqt", "_docked.pdbqt")
    log_name = ligand.replace(".pdbqt", "_vina.log")
    comp_base_name = ligand.replace('_expanded.pdbqt', '')

    print(f"→ Docking {comp_base_name}... (Calculating geometric conformations)")
    # Redirects stdout and stderr to the log to keep the console clean
    cmd = (
        f"./vina --receptor receptor.pdbqt --ligand \"{ligand}\" "
        f"--center_x {center_x} --center_y {center_y} --center_z {center_z} "
        f"--size_x {size_x} --size_y {size_y} --size_z {size_z} "
        f"--out {output_name} --exhaustiveness 8 > {log_name} 2>&1"
    )
    subprocess.run(cmd, shell=True)
    print(f"   ✓ Finished docking: {comp_base_name}")

print("\n==== ALL SIMULATIONS COMPLETED SUCCESSFULLY ====")

=== 🛠️ STEP 1: GENERATING 3D LIGAND STRUCTURES ===
   ✓ Successfully generated and verified: Baicalein_expanded.pdbqt
   ✓ Successfully generated and verified: Chrysin_expanded.pdbqt
   ✓ Successfully generated and verified: Oroxylin_A_expanded.pdbqt
   ✓ Successfully generated and verified: Exemestane_Control_expanded.pdbqt

=== 🚀 STEP 2: RUNNING PHYSICS-BASED DOCKING LOOP ===
Found 4 target ligands ready for simulation.

→ Docking Exemestane_Control... (Calculating geometric conformations)
   ✓ Finished docking: Exemestane_Control
→ Docking Baicalein... (Calculating geometric conformations)
   ✓ Finished docking: Baicalein
→ Docking Chrysin... (Calculating geometric conformations)
   ✓ Finished docking: Chrysin
→ Docking Oroxylin_A... (Calculating geometric conformations)
   ✓ Finished docking: Oroxylin_A

==== ALL SIMULATIONS COMPLETED SUCCESSFULLY ====


In [7]:
import os
import glob

print("Master Screening Evaluation (Phytochemicals vs. FDA Control):\n")
master_results = []
expanded_logs = glob.glob("*_expanded_vina.log")

for log_name in expanded_logs:
    if not os.path.exists(log_name) or os.path.getsize(log_name) == 0:
        continue
    with open(log_name, "r") as f:
        for line in f:
            parts = line.split()
            # Extracts the top-ranked mode binding energy line (Mode 1)
            if parts and parts[0] == "1":
                try:
                    affinity = float(parts[1])
                    # Clean up the name for the table display
                    comp_name = log_name.replace("_expanded_vina.log", "").replace("_", " ")
                    master_results.append((comp_name, affinity))
                    break
                except ValueError:
                    continue

# Sort from strongest binding (most negative thermodynamic energy) to weakest
master_results.sort(key=lambda x: x[1])

print(f"{'Compound Name':<25} | {'Binding Affinity (kcal/mol)':<30}")
print("-" * 60)
for name, score in master_results:
    print(f"{name:<25} | {score:<30} kcal/mol")

Master Screening Evaluation (Phytochemicals vs. FDA Control):

Compound Name             | Binding Affinity (kcal/mol)   
------------------------------------------------------------


In [8]:
!cat Exemestane_Control_expanded_vina.log

AutoDock Vina v1.2.5
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V

In [9]:
import subprocess
import glob
import os

print("=== 🛠️ STEP 1: FIXING THE RECEPTOR ===")
# The -xr flag forces OpenBabel to write a rigid receptor without ROOT/BRANCH tags
# The -p 7.4 flag protonates the protein at physiological pH
subprocess.run("obabel 3EQM.pdb -O receptor.pdbqt -p 7.4 -xr", shell=True)
print("✓ Receptor correctly formatted as a rigid structure!")

print("\n=== 🚀 STEP 2: RUNNING PHYSICS-BASED DOCKING LOOP ===")
center_x, center_y, center_z = 85.027, 54.737, 46.428
size_x, size_y, size_z = 22.0, 22.0, 22.0

expanded_hits = glob.glob("*_expanded.pdbqt")

for ligand in expanded_hits:
    output_name = ligand.replace(".pdbqt", "_docked.pdbqt")
    log_name = ligand.replace(".pdbqt", "_vina.log")
    comp_base_name = ligand.replace('_expanded.pdbqt', '')

    print(f"→ Docking {comp_base_name}...")
    cmd = (
        f"./vina --receptor receptor.pdbqt --ligand \"{ligand}\" "
        f"--center_x {center_x} --center_y {center_y} --center_z {center_z} "
        f"--size_x {size_x} --size_y {size_y} --size_z {size_z} "
        f"--out {output_name} --exhaustiveness 8 > {log_name} 2>&1"
    )
    subprocess.run(cmd, shell=True)
    print(f"   ✓ Finished docking: {comp_base_name}")

print("\n=== 📊 STEP 3: PARSING LOGS AND GENERATING MATRIX ===")
master_results = []
expanded_logs = glob.glob("*_expanded_vina.log")

for log_name in expanded_logs:
    if not os.path.exists(log_name) or os.path.getsize(log_name) == 0:
        continue
    with open(log_name, "r") as f:
        for line in f:
            parts = line.split()
            if parts and parts[0] == "1":
                try:
                    affinity = float(parts[1])
                    comp_name = log_name.replace("_expanded_vina.log", "").replace("_", " ")
                    master_results.append((comp_name, affinity))
                    break
                except ValueError:
                    continue

master_results.sort(key=lambda x: x[1])

print(f"\n{'Compound Name':<25} | {'Binding Affinity (kcal/mol)':<30}")
print("-" * 60)
for name, score in master_results:
    print(f"{name:<25} | {score:<30} kcal/mol")

=== 🛠️ STEP 1: FIXING THE RECEPTOR ===
✓ Receptor correctly formatted as a rigid structure!

=== 🚀 STEP 2: RUNNING PHYSICS-BASED DOCKING LOOP ===
→ Docking Exemestane_Control...
   ✓ Finished docking: Exemestane_Control
→ Docking Baicalein...
   ✓ Finished docking: Baicalein
→ Docking Chrysin...
   ✓ Finished docking: Chrysin
→ Docking Oroxylin_A...
   ✓ Finished docking: Oroxylin_A

=== 📊 STEP 3: PARSING LOGS AND GENERATING MATRIX ===

Compound Name             | Binding Affinity (kcal/mol)   
------------------------------------------------------------
Chrysin                   | 27.79                          kcal/mol
Baicalein                 | 31.56                          kcal/mol
Oroxylin A                | 37.05                          kcal/mol
Exemestane Control        | 68.57                          kcal/mol


In [10]:
import subprocess
import glob
import os

print("=== 🧹 STEP 1: EMPTYING THE BINDING POCKET ===")
# This extracts ONLY the protein atoms, deleting the trapped native ligand
subprocess.run('grep "^ATOM" 3EQM.pdb > clean_3EQM.pdb', shell=True)

# Convert the now-empty protein into a rigid receptor
subprocess.run("obabel clean_3EQM.pdb -O receptor.pdbqt -p 7.4 -xr", shell=True)
print("✓ Receptor cleaned, emptied, and formatted!")

print("\n=== 🚀 STEP 2: RUNNING PHYSICS-BASED DOCKING LOOP ===")
center_x, center_y, center_z = 85.027, 54.737, 46.428
size_x, size_y, size_z = 22.0, 22.0, 22.0

expanded_hits = glob.glob("*_expanded.pdbqt")

for ligand in expanded_hits:
    output_name = ligand.replace(".pdbqt", "_docked.pdbqt")
    log_name = ligand.replace(".pdbqt", "_vina.log")
    comp_base_name = ligand.replace('_expanded.pdbqt', '')

    print(f"→ Docking {comp_base_name} into the empty pocket...")
    cmd = (
        f"./vina --receptor receptor.pdbqt --ligand \"{ligand}\" "
        f"--center_x {center_x} --center_y {center_y} --center_z {center_z} "
        f"--size_x {size_x} --size_y {size_y} --size_z {size_z} "
        f"--out {output_name} --exhaustiveness 8 > {log_name} 2>&1"
    )
    subprocess.run(cmd, shell=True)
    print(f"   ✓ Finished docking: {comp_base_name}")

print("\n=== 📊 STEP 3: FINAL EVALUATION MATRIX ===")
master_results = []
expanded_logs = glob.glob("*_expanded_vina.log")

for log_name in expanded_logs:
    if not os.path.exists(log_name) or os.path.getsize(log_name) == 0:
        continue
    with open(log_name, "r") as f:
        for line in f:
            parts = line.split()
            if parts and parts[0] == "1":
                try:
                    affinity = float(parts[1])
                    comp_name = log_name.replace("_expanded_vina.log", "").replace("_", " ")
                    master_results.append((comp_name, affinity))
                    break
                except ValueError:
                    continue

# Sort from strongest binding (most negative) to weakest
master_results.sort(key=lambda x: x[1])

print(f"\n{'Compound Name':<25} | {'Binding Affinity (kcal/mol)':<30}")
print("-" * 60)
for name, score in master_results:
    print(f"{name:<25} | {score:<30} kcal/mol")

=== 🧹 STEP 1: EMPTYING THE BINDING POCKET ===
✓ Receptor cleaned, emptied, and formatted!

=== 🚀 STEP 2: RUNNING PHYSICS-BASED DOCKING LOOP ===
→ Docking Exemestane_Control into the empty pocket...
   ✓ Finished docking: Exemestane_Control
→ Docking Baicalein into the empty pocket...
   ✓ Finished docking: Baicalein
→ Docking Chrysin into the empty pocket...
   ✓ Finished docking: Chrysin
→ Docking Oroxylin_A into the empty pocket...
   ✓ Finished docking: Oroxylin_A

=== 📊 STEP 3: FINAL EVALUATION MATRIX ===

Compound Name             | Binding Affinity (kcal/mol)   
------------------------------------------------------------
Baicalein                 | -8.62                          kcal/mol
Exemestane Control        | -8.414                         kcal/mol
Chrysin                   | -8.379                         kcal/mol
Oroxylin A                | -7.789                         kcal/mol
